# 姿勢決定（Attitude Determination） 
いよいよ、**姿勢決定**の分野に足を踏み入れていく。ここでは、太陽の方向、磁場の向き、星の位置など、方向を表す複数の観測量を取得し、宇宙機の3次元姿勢を算出することに焦点を当てる。つまり、宇宙機座標系と基準座標系の両方で既知のベクトルを活用し、「宇宙機が基準座標系から見て、どの方向を向いているか？」を特定する手法である。

本記事では、姿勢決定分野の基礎となる古典的および最新のアルゴリズムを紹介する。扱う内容は以下の通り。

:::{admonition} 項目 
:class: note
- **TRIAD法**: 
  2つのベクトル観測に基づいて姿勢を算出する、基本的かつ直感的な手法。
- **Devenportのq-Method**: 
  コスト関数を最小化し、クォータニオンを回転表現として用いることで姿勢を推定する手法。
- **QUEST (QUaternion ESTimator)**: 
  q-Methodを最適化し、リアルタイム処理と高速計算に特化した手法。
- **OLAE (Optimal Linear Attitude Estimation)**: 
  ケーリー変換による線形化を用いて計算を簡略化し、特に複数の観測データを扱う際に有用な手法。
:::

本記事を通じて、これらの手法の仕組みを詳しく解説するとともに、計算効率や精度の観点から、それぞれの利点と欠点についても議論する。

:::{admonition} 目的 
:class: tip
- **一連の方位測定から姿勢を決定する**: 
  複数の方向観測を統合し、3次元空間で信頼性の高い姿勢を算出する方法を学ぶ。
- **姿勢決定に用いられる古典的および最新のアルゴリズムを解説する**:
  理論と実践の両面で使用される主要な手法について理解を深める。
- **剛体の基本的な姿勢座標特性を導出する**: 
  宇宙空間における剛体の回転運動を支配する数学的性質を習得する。
:::

In [8]:
# Import Relevant Libraries
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot

In [9]:
def quaternion_to_DCM(q):
    """
    Converts a quaternion to a direction cosine matrix (DCM).

    Args:
        q (np.array): A numpy array of size 4 (a row vector) representing the quaternion,
                      where q[0] is the scalar part (beta_0), and q[1], q[2], q[3] are the 
                      vector parts (beta_1, beta_2, beta_3).

    Returns:
        np.array: A 3x3 rotation matrix (DCM).

    Example:
        >>> q = np.array([1, 5, 6, 2])
        >>> quaternion_to_DCM(q)
        array([[ 0.38461538, -0.07692308,  0.91923077],
               [ 0.07692308,  0.99230769, -0.09615385],
               [-0.91923077,  0.09615385,  0.38461538]])
    """
    # Ensure q is a float array to maintain precision
    q = np.array(q, dtype=np.float64)
    
    # Check that the holonomic constraint of quaternion is satisfied, else normalize it
    q_norm = np.linalg.norm(q)
    if not np.isclose(q_norm, 1.0, atol=1e-8):
        q /= q_norm
    
    # Extract components
    q0, q1, q2, q3 = q
    
    # Compute the elements of the DCM
    C = np.array([
        [q0**2 + q1**2 - q2**2 - q3**2, 2 * (q1*q2 + q0*q3),         2 * (q1*q3 - q0*q2)],
        [2 * (q1*q2 - q0*q3),           q0**2 - q1**2 + q2**2 - q3**2, 2 * (q2*q3 + q0*q1)],
        [2 * (q1*q3 + q0*q2),           2 * (q2*q3 - q0*q1),           q0**2 - q1**2 - q2**2 + q3**2]
    ])
    
    return C

In [10]:
def DCM_to_quaternion(dcm):
    """
    Converts a Direction Cosine Matrix (DCM) to a quaternion using a method to ensure robustness against numerical issues.
    
    Args:
        dcm (np.array): A 3x3 rotation matrix (DCM).
    
    Returns:
        np.array: A quaternion represented as a numpy array of size 4, with the scalar component as the first element.
    
    Example Usage:
        >>> dcm = np.array([[-0.21212121, 0.96969697, 0.12121212],
                            [0.84848485, 0.12121212, 0.51515152],
                            [0.48484848, 0.21212121, -0.84848485]])
        >>> quaternion = DCM_to_quaternion(dcm)
        >>> print(quaternion)
    
    Notes:
    - If the scalar component (q0) is negative, it is flipped to positive. (Ensuring shortes path of rotation)
    - Corresponding vector component is also flipped when q0 is flipped. (Shepperd's Method)
    - Flipping maintains the quaternion's correct rotational encoding.
    - Ensures the quaternion represents a rotation of less than 180 degrees.
    - Adheres to quaternion algebra for accurate 3D rotation representation.
    """
    trace = np.trace(dcm)
    q_squared = np.zeros(4)
    q_squared[0] = (1.0 + trace) / 4.0
    q_squared[1] = (1.0 + 2 * dcm[0, 0] - trace) / 4.0
    q_squared[2] = (1.0 + 2 * dcm[1, 1] - trace) / 4.0
    q_squared[3] = (1.0 + 2 * dcm[2, 2] - trace) / 4.0

    q = np.zeros(4)
    max_index = np.argmax(q_squared)

    if max_index == 0:
        q[0] = np.sqrt(q_squared[0])
        q[1] = (dcm[1, 2] - dcm[2, 1]) / (4 * q[0])
        q[2] = (dcm[2, 0] - dcm[0, 2]) / (4 * q[0])
        q[3] = (dcm[0, 1] - dcm[1, 0]) / (4 * q[0])
    
    elif max_index == 1:
        q[1] = np.sqrt(q_squared[1])
        q[0] = (dcm[1, 2] - dcm[2, 1]) / (4 * q[1])
        if q[0] < 0:
            q[0] = -q[0]
            q[1] = -q[1]
        q[2] = (dcm[0, 1] + dcm[1, 0]) / (4 * q[1])
        q[3] = (dcm[2, 0] + dcm[0, 2]) / (4 * q[1])
        
    elif max_index == 2:
        q[2] = np.sqrt(q_squared[2])
        q[0] = (dcm[2, 0] - dcm[0, 2]) / (4 * q[2])
        if q[0] < 0:
            q[0] = -q[0]
            q[2] = -q[2]
        q[1] = (dcm[0, 1] + dcm[1, 0]) / (4 * q[2])
        q[3] = (dcm[1, 2] + dcm[2, 1]) / (4 * q[2])

    elif max_index == 3:
        q[3] = np.sqrt(q_squared[3])
        q[0] = (dcm[0, 1] - dcm[1, 0]) / (4 * q[3])
        if q[0] < 0:
            q[0] = -q[0]
            q[3] = -q[3]
        q[1] = (dcm[2, 0] + dcm[0, 2]) / (4 * q[3])
        q[2] = (dcm[1, 2] + dcm[2, 1]) / (4 * q[3])
    
    return q

# 4.1) 姿勢決定問題の概要
**姿勢決定**とは、姿勢センサの測定結果を利用して、基準座標系に対する物体の姿勢を算出することである。宇宙機を例に取ると、太陽方向ベクトル・恒星方向ベクトル・磁場ベクトルなど複数のベクトルが観測量として得られる。特定時刻に得られた複数の観測量から、3次元の姿勢を求める問題を**Deteriministic attitude estimation problem**と呼ぶ。一方で、時系列の観測量やレート測定（e.g. [ジャイロスコープ](https://ja.wikipedia.org/wiki/%E3%82%B8%E3%83%A3%E3%82%A4%E3%83%AD%E3%82%B9%E3%82%B3%E3%83%BC%E3%83%97)）を利用して、3次元の姿勢を求める問題を、**Dynamics attitude estimation problem**、**Attitude filter solution**などと呼ぶ。この問題を扱うには動的な手法（e.g. [カルマンフィルタ](https://ja.wikipedia.org/wiki/%E3%82%AB%E3%83%AB%E3%83%9E%E3%83%B3%E3%83%95%E3%82%A3%E3%83%AB%E3%82%BF%E3%83%BC)）は推定理論を必要とするため、別のトピックで扱うこととする。
本稿では、前者の**Deteriministic attitude estimation problem**、つまり同時刻に観測されたデータのみを用いて姿勢を静的に決定する問題を扱っていく。

姿勢決定の目的は、**慣性座標系における既知の基準ベクトル**と、**機体座標系で計測されたベクトル**を対応付けることで、宇宙機の3次元姿勢を算出することである。
姿勢は通常、**方向余弦行列（Direction Cosine Matrix, DCM）** $[\mathcal{B}\mathcal{N}]$ によって表され、この行列は慣性座標系のベクトル（$\mathcal{^N}\hat{v}$）を機体座標系のベクトル（$\mathcal{^B}\hat{v}$）へと変換する。

ここで、各センサの計測値は、宇宙機の3次元姿勢に関する**部分的な情報**しか提供しない。具体的には、1つのセンサ計測値が持つのは2自由度の情報であり、3自由度の姿勢を決定できない。つまり、完全な姿勢を特定するためには、**複数の独立した観測**が必要である。この問題では、**ノイズを含む機体座標系における複数のセンサ計測値**と、**正確で既知の慣性座標系の基準ベクトル**との対応関係を扱う。



**<ins>慣性座標系ベクトルと機体座標系ベクトル</ins>**

- **慣性座標系ベクトル（$\mathcal{^N}\hat{v}$）**：
  - これは、太陽の方向、恒星の方向、地磁場ベクトルなどのように、**慣性座標系における既知の参照方向**である。
  - 一般的に、これらの参照方向を決定するために必要な情報は2つである：
    1. **軌道上の位置**：GPSや地上局追跡によって取得。
    2. **軌道上の時刻**：オンボードクロックや地上との通信により取得。
  - これらの位置と時刻を用いて、太陽暦表（solar ephemeris）や[IGRF](https://www.ncei.noaa.gov/products/international-geomagnetic-reference-field)のような地磁場モデルなどの**精密な物理モデル**を通じて、慣性座標系内でのベクトルを算出する。

<br>

- **機体座標系ベクトル（$\mathcal{^B}\hat{v}$）**：
  - これは、宇宙機の機体に対して**相対的に観測された方向ベクトル**であり、宇宙機に搭載されたセンサから取得される。：
    - **サンセンサ**：太陽の方向を計測。
    - **磁気センサ**：地球の磁場を検出。
    - **スター・トラッカー（STT）**：恒星の位置を特定。
  - 各センサは、既知の参照天体を指し示す**機体座標系ベクトル**を提供する。



**<ins>慣性ベクトルと機体ベクトルの関係</ins>**

- 測定された機体座標系ベクトルと、既知の慣性座標系ベクトルの関係は、DCMで次のように表される。：
  
  $$
  \mathcal{^B}\hat{v}_k = [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k, \quad k = 1, \dots, N
  $$

  ここで：
  - $\mathcal{^N}\hat{v}_k$：位置・時刻・参照モデルから導かれる**慣性座標系ベクトル**
  - $\mathcal{^B}\hat{v}_k$：センサから得られる**機体座標系ベクトル**
  
　とする。なお、$\bar{\mathcal{B}}$は、推定された機体座標系であることを表す。

- 逆に、DCMの転置行列を用いれば、機体座標系から慣性座標系への変換が可能となる：

  $$
  \mathcal{^N}\hat{v}_k = [\bar{\mathcal{B}}\mathcal{N}]^T   \mathcal{^B}\hat{v}_k
  $$


# 4.2) TRIAD法

**[TRIAD法](https://en.wikipedia.org/wiki/Triad_method)**は **2つのベクトル観測**を用いて、特定の時刻における宇宙機の姿勢を推定する**決定論的手法**である。
この手法では、問題を簡略化するために、**中間座標系（TRIAD座標系）** を導入する。  
このTRIAD座標系を、慣性座標系（例：恒星の方向や地磁場）および機体座標系（センサによる観測値）にそれぞれ適用することで、姿勢推定を行列計算として単純に解くことが可能となる。

以下に、この手法の特徴と主要なステップを示す。

**<ins>TRIAD法の特徴</ins>**

- **シンプルで効率的**：  
  TRIAD法は計算負荷が非常に低いため、処理能力が限られたミッションでも使用しやすい手法である。

- **限られた観測値でも信頼性が高い**：  
  太陽方向や地磁場のように、**2つの観測ベクトルしか得られない場合**でも、迅速に姿勢を推定する手段として有効。一方で、3つ以上の観測ベクトルは扱えない。

- **観測値の最適な使い分け**：  
  TRIAD法では、最も精度の高い観測（通常は太陽方向）を**主参照軸（primary axis)**として使用し、磁場方向のような精度が劣る観測値は**補助的な参照軸**として扱える。

**<ins>TRIAD法のStep</ins>**


**1. 座標系の定義**

- **慣性座標系（Inertial Frame）**：  
  $\mathcal{N}$フレームで表され、太陽の位置や地磁場など、宇宙空間における既知の方向を表す。これらの方向は、通常、衛星の地球周回軌道上の位置・時刻が分かっているため、既知と見なされる。

- **機体座標系（Body Frame）**：  
  $\mathcal{B}$フレームで表され、太陽センサや磁気センサなどの観測によって得られた、機体に固定された座標系での測定値を表す。（センサ座標系から機体座標系への変換は完了していることが前提）

- **TRIAD座標系（Triad Frame）**：  
  中間座標系であり、$\mathcal{T}$フレームで表される。センサノイズの影響を軽減するために導入される。ノイズの多いセンサデータから直接機体の姿勢を求めると誤差が生じやすいため、このTRIAD座標系を媒介とすることで、より安定した姿勢推定が可能となる。

<br>

**2. 第1軸の定義**

- 最も精度の高い測定値を選び、TRIAD座標系の第1軸 $\hat{t}_1$ とする。  
  通常、太陽センサの方が、地磁場の変動による影響を受けやすい磁気センサよりも高精度となる。

- 機体座標系における太陽ベクトルを $\hat{s}_b$、慣性座標系における太陽ベクトルを $\hat{s}_n$ とすると：

  $$
  \hat{t}_1 = \hat{s}
  $$

<br>

**3. 第2軸の直交ベクトル計算**

- 補助的な測定量を使用して、第1軸に直交する第2軸を定義する。

- 例えば、機体座標系での地磁場を $\hat{m}_b$、慣性座標系での地磁場を $\hat{m}_n$ とすると：

  $$
  \hat{t}_2 = \frac{\hat{s} \times \hat{m}}{|\hat{s} \times \hat{m}|}
  $$

- $\hat{t}_2$ は単位ベクトルとして正規化される。

<br>

**4. 第3軸の定義**

- TRIAD座標系を完成させるために、第1軸と第2軸の外積から第3軸を定義する：

  $$
  \hat{t}_3 = \hat{t}_1 \times \hat{t}_2
  $$

<br>

**5. 変換行列の構成**

- TRIAD座標系の各軸は、機体座標系と慣性座標系の両方で以下のように構成される。

  **機体座標系のTRIADベクトル**：

  $$
  \mathcal{^B}\hat{t}_1 = \mathcal{^B}\hat{s}
  $$
  $$
  \mathcal{^B}\hat{t}_2 = \frac{\mathcal{^B}\hat{s} \times \mathcal{^B}\hat{m}}{|\mathcal{^B}\hat{s} \times \mathcal{^B}\hat{m}|}
  $$
  $$
  \mathcal{^B}\hat{t}_3 = \mathcal{^B}\hat{t}_1 \times \mathcal{^B}\hat{t}_2
  $$

  **慣性座標系のTRIADベクトル**：

  $$
  \mathcal{^N}\hat{t}_1 = \mathcal{^N}\hat{s}
  $$
  $$
  \mathcal{^N}\hat{t}_2 = \frac{\mathcal{^N}\hat{s} \times \mathcal{^N}\hat{m}}{|\mathcal{^N}\hat{s} \times \mathcal{^N}\hat{m}|}
  $$
  $$
  \mathcal{^N}\hat{t}_3 = \mathcal{^N}\hat{t}_1 \times \mathcal{^N}\hat{t}_2
  $$

- センサ計測誤差がない場合、両フレームにおけるトライアド表現は一致する。

- **機体→TRIAD座標系**（$[\bar{\mathcal{B}}\mathcal{T}]$）と**慣性TRIAD座標系**（$[\mathcal{N}\mathcal{T}]$）を構成する：

  $$
  [\bar{\mathcal{B}}\mathcal{T}] = \begin{bmatrix} \mathcal{^B}\hat{t}_1 & \mathcal{^B}\hat{t}_2 & \mathcal{^B}\hat{t}_3 \end{bmatrix}
  $$
  $$
  [\mathcal{N}\mathcal{T}] = \begin{bmatrix} \mathcal{^N}\hat{t}_1 & \mathcal{^N}\hat{t}_2 & \mathcal{^N}\hat{t}_3 \end{bmatrix}
  $$

<br>

**6. 姿勢行列の計算**

- 最後に、慣性座標系から機体座標系への変換を表す姿勢行列 $[\bar{\mathcal{B}}\mathcal{N}]$ を計算する：

  $$
  \bar{\mathcal{B}}\mathcal{N} = [\bar{\mathcal{B}}\mathcal{T}] \cdot [\mathcal{N}\mathcal{T}]^T
  $$

- このDCMから、必要な姿勢パラメータ（オイラー角、MRPなど）を抽出できる。

<br>

**7. 推定誤差の評価**

- 実際の姿勢行列（$[\mathcal{B}\mathcal{N}]$）が既知であれば、推定誤差行列は次のように計算される：

  $$
  [\bar{\mathcal{B}}\mathcal{B}] = [\bar{\mathcal{B}}\mathcal{N}] \cdot ([\mathcal{B}\mathcal{N}])^T
  $$

- この $[\bar{\mathcal{B}}\mathcal{B}]$ は誤差行列を表し、推定が完全であれば単位行列となる。

- 誤差の大きさは、**軸-角表現(axis-angle representation)**を使って回転軸と角度を抽出することで評価できる。これにより、推定された機体座標系が真の座標系からどれだけ回転しているか（ラジアンまたは度数）を定量的に把握できる。

- この処理により、姿勢推定の精度を高い精度で評価することができる。


**TRIAD法**は姿勢決定における基本的な手法であり、2つのセンサ測定値を直接使用できるという点で非常にシンプルかつ有用である。特に、利用可能なデータが限られている状況において、迅速かつ信頼性の高い姿勢推定を実現するための、航空宇宙工学における基礎的ツールとなる。
一方で、3つ以上の観測量を扱えないというデメリットがあり、任意の数の観測量を扱える**q-Method**, **QUEST**, **OLAE**を導入する動機となる。

In [4]:
def Triad(b1, b2, r1, r2):
    """
    TRIAD algorithm.
    
    Inputs:
        b1, b2 : First and second unit vectors in spacecraft body frame (shape: (3,))
        r1, r2 : Corresponding vectors in the reference (inertial) frame (shape: (3,))
    
    Outputs:
        C : Rotation matrix (3x3) that transforms vectors in the inertial frame 
            to the body frame.
        q : Quaternion (numpy array of shape (4,)) representing the rotation 
            from the inertial frame to the body frame.
            
    Note:
        b1 is assumed to be much more accurately determined than b2.
        Thus, the estimation satisfies b1 = Q * r1 exactly, while b2 = Q * r2 only approximately.
    """
    # normalization
    r1 /= np.linalg.norm(r1)
    r2 /= np.linalg.norm(r2)
    b1 /= np.linalg.norm(b1)
    b2 /= np.linalg.norm(b2)
    
    # Construct the inertial-frame TRIAD vectors
    t1_r = r1
    cross_r1_r2 = np.cross(r1, r2)
    t2_r = cross_r1_r2 / np.linalg.norm(cross_r1_r2)
    t3_r = np.cross(t1_r, t2_r)
    
    # Construct the body-frame TRIAD vectors (estimated)
    t1_b = b1
    cross_b1_b2 = np.cross(b1, b2)
    t2_b = cross_b1_b2 / np.linalg.norm(cross_b1_b2)
    t3_b = np.cross(t1_b, t2_b)

    # Assemble the TRIAD matrices
    B_bar_T = np.column_stack((t1_b, t2_b, t3_b))  # Estimated body frame TRIAD matrix
    N_T = np.column_stack((t1_r, t2_r, t3_r))      # Inertial frame TRIAD matrix
    
    # Compute the rotation matrix from inertial frame to body frame
    B_bar_N = np.matmul(B_bar_T, N_T.T)
    
    return B_bar_N

In [6]:
# Concept Check 2 - TRIAD Method, Q1
b1 = np.array([0.8190, -0.5282, 0.2242])
b2 = np.array([-0.3138, -0.1584, 0.9362])
r1 = np.array([1.0, 0.0, 0.0])
r2 = np.array([0.0, 0.0, 1.0])
    
B_bar_N = Triad(b1, b2, r1, r2)
print("Rotation Matrix BN:")
print(B_bar_N)

Rotation Matrix BN:
[[ 0.81899104  0.45928237 -0.34396712]
 [-0.52819422  0.83763943 -0.13917991]
 [ 0.22419755  0.29566855  0.92860948]]


In [7]:
# Concept Check 2 - TRIAD Method, Q2
B_bar_B = np.matmul(B_bar_N, B_bar_N.T)
print("Rotation Matrix B_bar_B:")
print(B_bar_B)
#axis, phi = DCM_to_PRV(B_bar_B)
#print(phi)

Rotation Matrix B_bar_B:
[[1.00000000e+00 5.22634982e-17 1.06527965e-18]
 [5.22634982e-17 1.00000000e+00 6.86536202e-17]
 [1.06527965e-18 6.86536202e-17 1.00000000e+00]]


# 4.3) Wahba's Problem 

**<ins>Wahba's Problemとは</ins>**

- **目的**:  
  センサで計測されたボディ座標系のベクトルと、既知の慣性座標系の対応するベクトルとの誤差を最小化することで、宇宙機の最適な姿勢を求めることである。
  この姿勢は、方向余弦行列（Direction Cosine Matrix, DCM） $[\bar{\mathcal{B}}\mathcal{N}]$ により表され、慣性座標系のベクトル（$\mathcal{^N}\hat{v}$）とボディ座標系のベクトル（$\mathcal{^B}\hat{v}$）との対応関係を記述する。

- **課題**:  
  センサの測定値はノイズを含み、宇宙機の3次元姿勢は一意に決定されない。
  そのため、センサノイズや変動を考慮しながら、測定されたベクトルと既知のベクトルとの不一致を最小化する $[\bar{\mathcal{B}}\mathcal{N}]$ を求める必要がある。


**<ins>定式化</ins>**

1. **慣性座標系からボディ座標系への写像**:

   $$
   \mathcal{^B}\hat{v}_k = [\bar{\mathcal{B}}\mathcal{N}] \, \mathcal{^N}\hat{v}_k, \quad k = 1, \dots, N
   $$

   ここで：  
   - $\mathcal{^N}\hat{v}_k$: モデルや天体暦データ（例：太陽の方向、地磁気）などから得られる**慣性座標系**の既知の参照ベクトル。  
   - $\mathcal{^B}\hat{v}_k$: 宇宙機のオンボードセンサ（例：太陽センサ、磁気センサ）によって**ボディ座標系**で測定された対応するベクトル。  
   - $N$: 利用可能なベクトルペア（観測）の数。

<br>

2. **誤差最小化**:  
   Wahbaは、姿勢決定問題を解くための最小二乗法アプローチを提案した。  
   その目的は、以下のコスト関数を最小化する $[\bar{\mathcal{B}}\mathcal{N}]$ を求めることである。：

   $$
   J([\bar{\mathcal{B}}\mathcal{N}]) = \frac{1}{2} \sum_{k=1}^N w_k \| \mathcal{^B}\hat{v}_k - [\bar{\mathcal{B}}\mathcal{N}] \, \mathcal{^N}\hat{v}_k \|^2
   $$

   ここで：
   - $w_k$: 各観測に割り当てられる重み（精度や信頼性を反映）。  
   - $\|\cdot\|$: ユークリッドノルム。  

   $\frac{1}{2}$の定数は、最適化の際に微分計算を簡略化するために導入されている（2で打ち消し合うため）。

<br>

3. **完全な測定値の場合**:  
   すべての測定がノイズフリーであれば、コスト関数は以下のようになりゼロとなる。：

   $$
   J = 0
   $$

---

**<ins>なぜ複数の測定が必要か？</ins>**

- 各測定ベクトルは、例えば方位角と仰角のように、**独立した2つの情報しか提供しない**。  
  そのため：
  - 単一のベクトルでは、3次元の姿勢（自由度3）を完全に特定出来ない。  
  - 姿勢行列を一意に求めるためには、少なくとも**2つの独立した参照ベクトル**が必要である。  
  - $N > 2$の場合、問題は過剰指定（over-specified）となり、最小二乗法が有効となる。


**<ins>最小二乗法との類似性</ins>**

- コスト関数 $J([\bar{\mathcal{B}}\mathcal{N}])$ は、最小二乗回帰問題に類似している：
  - 回帰問題では、予測値と観測値の二乗誤差を最小化する。  
  - Wahbaの問題では、ボディ座標系で測定されたベクトルと、その対応する慣性座標系の参照ベクトルとの二乗誤差を最小化する。

このため、DCM $[\bar{\mathcal{B}}\mathcal{N}]$ は、参照ベクトルと観測値を最もよく整合させる**最適な回転行列**（best-fit rotation matrix）となる。


**<ins>重みの重要性</ins>**

- センサごとに測定精度は異なる：
  - **太陽センサ**：通常、より高精度かつ信頼性が高い。  
  - **磁力計（MTM）**：環境変動の影響でノイズが多くなりがち。  
- 各センサの信頼性を反映するために、重み $w_k$ が割り当てられる。  
- すべての重み $w_k$ を一様にスケールしても、最適化は相対的な問題であるため、結果には影響ないことに留意する。（例えば、$w_1=1, w_2=2$と、$w_1=2, w_2=4$とで、得られる結果は変わらない）


**<ins>推定精度の評価</ins>**

1. **推定誤差の計算**:  
   推定された DCM $[\bar{\mathcal{B}}\mathcal{N}]$ を、もし既知であれば真の DCM $[\mathcal{B}\mathcal{N}]$ と比較する：

   $$
   [\mathcal{B}\bar{\mathcal{B}}] = [\bar{\mathcal{B}}\mathcal{N}] \cdot [\mathcal{B}\mathcal{N}]^T
   $$

   - $[\mathcal{B}\bar{\mathcal{B}}]$ は、推定された姿勢と真の姿勢とのずれ（誤差DCM）を定量化する。

<br>

2. **軸−角表現による誤差**:  
   - 推定が完全な場合：

     $$
     [\mathcal{B}\bar{\mathcal{B}}] = {I}
     $$
     
   - それ以外の場合は、軸−角表現を用いて回転軸と回転角度を算出し、回転角度が推定誤差の大きさを示す。

---

**<ins>まとめ</ins>**

Wahbaの問題は、姿勢決定を最小二乗最適化問題として定式化するものである。  
その基本的なアイデアは、DCM $[\bar{\mathcal{B}}\mathcal{N}]$ を用いて、測定されたボディ座標系のベクトルと写像された慣性座標系のベクトルとの誤差のノルム（大きさ）を求めることにある。

- ノルムが**ゼロ**であれば、完璧な整合性を示し、DCM $[\bar{\mathcal{B}}\mathcal{N}]$ は慣性ベクトルを正確にボディ座標系に写像していることになり、$J = 0$となる。  
- ノルムが**ゼロでない**場合、センサノイズや不正確さにより写像に誤差が生じ、$J > 0$となります。

Wahbaの問題は最小二乗法の考え方を姿勢決定に拡張したものであり、目的は誤差ノルムを最小化して最適なDCMを見つけることにある。


# 4.4) Devenport's q-Method

Devenportのq-Methodは、クォータニオンを用いて宇宙機の姿勢を決定するための強力な手法である。
この手法は、Wahbaの問題を固有値問題に帰着させることで、効率的に最適なクォータニオン―すなわち宇宙機の姿勢を最もよく記述するクォータニオン―を求めるものである。

**<ins>目的</ins>**

- クォータニオン $\beta = [\beta_0, \beta_1, \beta_2, \beta_3]^T$ の最尤推定値を求めることが目的である。ここで：
  - $\beta_0$ はスカラー部。
  - $\beta_1, \beta_2, \beta_3$ はベクトル部を形成する。

- クォータニオン $\beta$ は **単位ベクトル** であり、$\|\beta\|^2 = 1$ を満たすことで有効な回転を表す。

- この手法では Wahba のコスト関数を最小化を考える：

  $$
  J([\bar{\mathcal{B}}\mathcal{N}]) = \frac{1}{2} \sum_{k=1}^N w_k \| \mathcal{^B}\hat{v}_k - [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \|^2
  $$

- $\mathcal{^B}\hat{v}_k$ および $\mathcal{^N}\hat{v}_k$ は 軸 $\mathcal{i,j,k}$ における単位ベクトルである。



**<ins>Wahbaのコスト関数の再定式化とゲイン関数の導出</ins>**

 1. **Wahbaのコスト関数の再定式化**:
- Wahbaのコスト関数は、ボディ座標系ベクトル $\mathcal{^B}\hat{v}_k$ と、DCM $[\bar{\mathcal{B}}\mathcal{N}]$ によって変換された慣性座標系ベクトル $\mathcal{^N}\hat{v}_k$ の誤差を表す：

  $$
  J([\bar{\mathcal{B}}\mathcal{N}]) = \frac{1}{2} \sum_{k=1}^N w_k \| \mathcal{^B}\hat{v}_k - [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \|^2
  $$

- このコスト関数は次のように書き直せる：

  $$
  J = \frac{1}{2} \sum_{k=1}^N w_k \left( \mathcal{^B}\hat{v}_k - [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \right)^T \left( \mathcal{^B}\hat{v}_k - [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \right)
  $$

- 内積を展開すると：

  $$
  J = \frac{1}{2} \sum_{k=1}^N w_k \left( (\mathcal{^B}\hat{v}_k)^T (\mathcal{^B}\hat{v}_k) + (\mathcal{^N}\hat{v}_k)^T [\bar{\mathcal{B}}\mathcal{N}]^T [\bar{\mathcal{B}}\mathcal{N}] (\mathcal{^N}\hat{v}_k) - 2 (\mathcal{^B}\hat{v}_k)^T [\bar{\mathcal{B}}\mathcal{N}] (\mathcal{^N}\hat{v}_k) \right)
  $$

- 単位ベクトルの性質より：

  $$
  (\mathcal{^B}\hat{v}_k)^T (\mathcal{^B}\hat{v}_k) = 1, \quad (\mathcal{^N}\hat{v}_k)^T (\mathcal{^N}\hat{v}_k) = 1
  $$

  さらに、

  $$
  [\bar{\mathcal{B}}\mathcal{N}]^T [\bar{\mathcal{B}}\mathcal{N}] = I
  $$

  これにより、式は次のように単純化される：

  $$
  J([\bar{\mathcal{B}}\mathcal{N}]) = \frac{1}{2} \sum_{k=1}^N w_k \left( 2 - 2 \mathcal{^B}\hat{v}_k^T [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \right)
  $$

- 最後に 2 を因数として取り出すと：

  $$
  J([\bar{\mathcal{B}}\mathcal{N}]) = \sum_{k=1}^N w_k \left( 1 - \mathcal{^B}\hat{v}_k^T [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \right)
  $$



2. **ゲイン関数**:
- $J$ を最小化することは、次を最大化することに等しい：

  $$
  g = \sum_{k=1}^N w_k \mathcal{^B}\hat{v}_k^T [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k
  $$

- ゲイン関数 $g$ は、ボディ座標系ベクトルと慣性座標系ベクトルの整合度を表す。



**<ins>クォータニオンによるゲイン関数の表現</ins>**

1. **DCM $[\bar{\mathcal{B}}\mathcal{N}]$ をクォータニオンで表現**:

$$
[\bar{\mathcal{B}}\mathcal{N}] = (\beta_0^2 - \epsilon^T \epsilon)[I_{3 \times 3}] + 2\epsilon\epsilon^T - 2\beta_0[\tilde{\epsilon}]
$$

- ここで：
  - $\epsilon = [\beta_1, \beta_2, \beta_3]^T$ はクォータニオンのベクトル部。
  - $[\tilde{\epsilon}]$ は $\epsilon$ のスキュー対称行列。

2. **$g$ の簡略化**:
- 上記の DCM を $g$ に代入すると：

  $$
  g(\beta) = \beta^T [K] \beta
  $$

- ここで $[K]$ は**ゲイン行列**である。


**<ins>ゲイン行列 $[K]$ の構成</ins>**

1. **姿勢プロファイル行列 $[B]$ の計算**：

$$
[B] = \sum_{k=1}^N w_k \mathcal{^B}\hat{v}_k (\mathcal{^N}\hat{v}_k)^T
$$
 
- $[B]$ は測定されたボディ座標系ベクトルと既知の慣性座標系ベクトルの関係を示す行列であり、それらの回転整合性を重み付きで要約している。

1. **$\sigma$, $[Z]$, および $[S]$ の計算**:

- $\sigma = \text{tr}([B])$：$[B]$ のトレース（対角要素の和）
- $[Z] = [B_{23} - B_{32}, B_{31} - B_{13}, B_{12} - B_{21}]^T$
- $[S] = [B] + [B]^T$

3. **ゲイン行列 $[K]$ の構築**:

$$
[K] = \begin{bmatrix} 
\sigma & Z^T \\ 
Z & [S] - \sigma [I_{3 \times 3}] 
\end{bmatrix}
$$


**<ins>制約付き最適化</ins>**

1. **単位クォータニオンの制約**:

$$
\|\beta\|^2 = 1 \quad \Rightarrow \quad \beta^T\beta = 1
$$

2. **ラグランジュ乗数法による最適化**:

- 制約を満たしつつ $g(\beta)$ を最大化するため、ラグランジュ乗数 $\lambda$ を導入する。

- ラグランジュ関数は：

$$
g'(\beta) = \beta^T[K]\beta - \lambda(\beta^T\beta - 1)
$$

- $\beta$ について微分し、停留点を求めると：

$$
\frac{\partial g'}{\partial \beta} = 2[K]\beta - 2\lambda\beta = 0
$$

- 整理すると：

$$
[K]\beta = \lambda\beta
$$

- よって、$\beta$ は $[K]$ の固有ベクトルであり、$\lambda$ はその固有値である。

**<ins>最大固有値</ins>**

代入すると、最適なクォータニオン  $\beta$ に対応するのは、行列 $[K]$の最大固有値であることが分かる。つまり、行列 $[K]$の最大固有値に対応する固有ベクトルが、求める最適なクォータニオン  $\beta$ である。


**<ins>解法手順</ins>**

1. 観測データから $[B]$ 行列を計算。
2. $[K]$ 行列を構築。
3. 固有値問題 $[K]\beta = \lambda\beta$ を解く。
4. 最大の固有値に対応する固有ベクトルを選択。これが最適なクォータニオン $\beta$ となる。



**<ins>まとめ</ins>**

DavenportのQ-Methodは、Wahbaの問題を解くための強力で体系的な静的姿勢決定手法である：

1. **Wahbaの問題**：測定されたベクトルと基準ベクトルのずれを定量化し、その不一致を最小化することを目的とする。

2. **線形代数の活用**：誤差の2乗を内積に変換し、最適化対象であるゲイン関数 $g(\beta)$ を導出する。

3. **クォータニオン表現**：$g(\beta) = \beta^T K \beta$ により、クォータニオンを使った最適化が可能になる。

4. **ゲイン行列 $K$ の構成**：姿勢プロファイル行列 $B$ をもとに $K$ を計算する。

5. **制約付き最適化**：単位クォータニオン制約を満たすためにラグランジュ乗数を用いる。

6. **固有値問題として解く**：最大の固有値に対応する固有ベクトルが、最適なクォータニオンとなる。

7. **最適なDCMの構成**：得られたクォータニオンをもとに、最良の回転行列 $[\bar{\mathcal{B}}\mathcal{N}]$ を導出する。

このようにして、DavenportのQ-Methodは線形代数・クォータニオン・最適化理論を融合させた洗練された姿勢推定アルゴリズムとなっている。ステップ6での固有値計算は計算コストが高いため、リアルタイム用途には不向きな場合もあるが、高精度な姿勢推定の基礎として広く活用されている。


In [11]:
import numpy as np

def Q_method(v_b, v_i, w):
    """
    Davenport's q-method

    Parameters:
        v_b (ndarray): 3xN unit vectors in the body frame
        v_i (ndarray): 3xN corresponding unit vectors in the inertial frame
        w (ndarray): Nx1 non-negative weights for each observation

    Returns:
        C_opt (ndarray): 3x3 Optimal rotation matrix from inertial to body frame
        q_opt (ndarray): 4x1 Optimal quaternion [q1, q2, q3, q0]
    """

    # Attitude profile matrix B
    B = (v_b * w.T) @ v_i.T

    # Z vector
    Z = np.array([
        B[1, 2] - B[2, 1],
        B[2, 0] - B[0, 2],
        B[0, 1] - B[1, 0]
    ])

    # Gain matrix K
    S = B + B.T
    sigma = np.trace(B)
    K = np.zeros((4, 4))
    K[1:4, 1:4] = S - np.eye(3) * sigma
    K[0, 1:4] = Z
    K[1:4, 0] = Z.T
    K[0, 0] = sigma

    # Eigen-decomposition
    eigenvalues, eigenvectors = np.linalg.eig(K)
    max_index = np.argmax(eigenvalues)
    q_opt = eigenvectors[:, max_index]

    # Normalize quaternion
    q_opt = q_opt / np.linalg.norm(q_opt)

    # Convert quaternion to DCM
    C_opt = quaternion_to_DCM(q_opt)

    return C_opt, q_opt

In [12]:
# Concept Check 3, 4 - Devenport's q-method
v_b = np.array([
    [0.8273, 0.5541, -0.0920],
    [-0.8285, 0.5522, -0.0955]
])

v_i = np.array([
    [-0.1517, -0.9669, 0.2050],
    [-0.8393, 0.4494, -0.3044]
])


w = np.array([1,1])

# Call the Davenport Q-Method function
C_opt, q_opt = Q_method(v_b.T, v_i.T, w)

# Print the resulting Direction Cosine Matrix (DCM)
print("Estimated Direction Cosine Matrix (DCM):")
print(C_opt)

Estimated Direction Cosine Matrix (DCM):
[[ 0.41593634 -0.85489355  0.31008704]
 [-0.83375669 -0.49463674 -0.24532484]
 [ 0.36310707 -0.15649763 -0.91851061]]


# 4.5) QUEST

**<ins>目的</ins>**

- QUEST（**QUaternion ESTimator**）法は、Davenport の Q-Method よりも高速に Wahba 問題を解く手法である。計算コストの高い固有値・固有ベクトル問題を直接解くことを避けている。

- 目標はクォータニオン $\beta = [\beta_0, \beta_1, \beta_2, \beta_3]^T$ を求めることである。ここで：
  - $\beta_0$ はスカラー成分、
  - $\beta_1, \beta_2, \beta_3$ はベクトル成分
  
  を構成する。

- クォータニオン $\beta$ は**単位ベクトル**であり、$\|\beta\|^2 = 1$ を満たして有効な回転を表す。

- Davenport の Q-Method と同様に、QUEST は Wahba のコスト関数を最小化する：

  $$
  J([\bar{\mathcal{B}}\mathcal{N}]) = \frac{1}{2} \sum_{k=1}^N w_k \| \mathcal{^B}\hat{v}_k - [\bar{\mathcal{B}}\mathcal{N}] \mathcal{^N}\hat{v}_k \|^2
  $$


**<ins>Wahba のコスト関数の書き換えと最適固有値の導入</ins>**

1. **コスト関数とゲイン関数の関連付け**：
   - コスト関数 $J$ は、ゲイン関数 $g$ を用いて以下のように書き換えられる：

     $$
     J = \sum_{k=1}^N w_k - g
     $$

   - Davenport の Q-Method において、最適なクォータニオン $\bar\beta$ に対するゲイン関数 $g$ はゲイン行列 $[K]$ の最大固有値 $\lambda_{\text{opt}}$ に等しい：

     $$
     g(\bar\beta) = \lambda_{\text{opt}}
     $$

2. **簡略化されたコスト関数**：
   - $g(\bar\beta) = \lambda_{\text{opt}}$ を代入すると：

     $$
     J = \sum_{k=1}^N w_k - \lambda_{\text{opt}}
     $$

3. **最適固有値の表現**：
   - 上式を $\lambda_{\text{opt}}$ について整理すると：

     $$
     \lambda_{\text{opt}} = \sum_{k=1}^N w_k - J
     $$

   - 観測ノイズが小さい（QUEST の仮定）場合、$J$ は 0 に近いため：

     $$
     \lambda_{\text{opt}} \approx \sum_{k=1}^N w_k
     $$

---

**<ins>数値最適化：根の求解によるアプローチ</ins>**

1. **固有値問題の回避**：
   - ゲイン行列 $[K]$ の固有値問題を直接解く代わりに、QUEST は**反復的な根の求解法**（例：ニュートン・ラフソン法）を使って $\lambda_{\text{opt}}$ を近似する。

2. **特性方程式**：
   - ゲイン行列の固有値は、以下の特性方程式の根である：

     $$
     f(s) = \text{det}([K] - s[I_{4 \times 4}]) = 0
     $$

3. **ニュートン・ラフソン法**：
   - 最大の根 $\lambda_{\text{opt}}$ を求めるために、ニュートン・ラフソン法で推定値を反復的に更新する：

     $$
     \lambda_{i+1} = \lambda_i - \frac{f(\lambda_i)}{f'(\lambda_i)}
     $$

   - 初期値として $\lambda_0 = \sum_{k=1}^N w_k$ を使うことで、効率よく最大固有値に収束する。



**<ins>Rodrigues パラメータと姿勢の再構築</ins>**

1. **古典的 Rodrigues パラメータ（CRPs）**：
   - クォータニオンの表現を簡略化するため、QUEST は**Rodrigues パラメータベクトル** $\mathbf{q}$ を導入する：

     $$
     \mathbf{q} = \frac{\epsilon}{\beta_0} = \begin{bmatrix} \beta_1 / \beta_0 \\ \beta_2 / \beta_0 \\ \beta_3 / \beta_0 \end{bmatrix}
     $$

     ここで、$\epsilon = [\beta_1, \beta_2, \beta_3]^T$ はクォータニオンのベクトル部分である。

2. **線形化された固有ベクトル問題**：
   - 固有ベクトル問題 $[K] \beta = \lambda_{\text{opt}} \beta$ は以下のように書き換えられる：

     $$
     \left([S] - \lambda_{\text{opt}} [I_{3 \times 3}] + \sigma\right) \mathbf{q} = [Z]
     $$

   - ここで、$[S]$, $[Z]$, $\sigma$ は姿勢プロファイル行列 $[B]$ に由来する。

3. **CRPs の解法**：
   - 以下を解いて $\mathbf{q}$ を求める：

     $$
     \mathbf{q} = \left((\lambda_{\text{opt}} + \sigma)[I_{3 \times 3}] - [S]\right)^{-1} [Z]
     $$

4. **クォータニオンの再構成**：
   - $\mathbf{q}$ からクォータニオンを再構築する式は以下の通り：

     $$
     \beta = \frac{1}{\sqrt{1 + \mathbf{q}^T \mathbf{q}}} \begin{bmatrix} 1 \\ \mathbf{q} \end{bmatrix}
     $$



**<ins>QUEST の利点</ins>**

1. **高速性**：
   - 固有値分解を直接行わず、リアルタイム用途においても高速に動作する。

2. **反復精度**：
   - ニュートン・ラフソン法などの反復法を通じて高精度な解を得られる。

3. **柔軟性**：
   - 多様な重み付けや複数ベクトルの観測に対応できる。



**<ins>まとめ</ins>**

QUEST 法は Davenport の Q-Method を基礎として、より効率的に Wahba 問題を解決する：

1. **Wahba のコスト関数**：コスト関数 $J$ を最小化しつつ、ゲイン関数 $g$（$[K]$ の最大固有値）を最大化する。

2. **固有値分解の回避**：ニュートン・ラフソン法を用いて $\lambda_{\text{opt}}$ を反復的に求めることで、直接的な固有値分解を回避。

3. **姿勢表現**：古典的 Rodrigues パラメータ（CRPs）を用いてクォータニオン表現を簡素化し、最適なクォータニオン $\beta$ を再構築。

4. **実用的な利点**：数値的な高速性と高精度を兼ね備えており、リアルタイムな宇宙機の姿勢決定に最適。

QUEST は、Davenport の Q-Method を精緻に改良した手法であり、精度を維持しながら計算効率を大幅に向上させている。

In [13]:
def quest(v_b, v_i, w):
    """
    Quaternion Estimator (QUEST) according to Shuster (1981).

    Inputs:
      v_b : 3 x n numpy array
            Unit measurement vectors in the spacecraft body frame.
      v_i : 3 x n numpy array
            Corresponding unit vectors known in the inertial frame.
      w   : n-element 1D numpy array
            Non-negative weights assigned to each observation.

    Outputs:
      C_opt : 3 x 3 Optimal rotation matrix that transforms vectors in 
              inertial frame to vectors in body frame.
      q_opt : 4-element optimal quaternion [q0, q1, q2, q3] (unit quaternion)
              that transforms vectors in inertial frame to vectors in body frame.
    """
    # Error tolerance
    tolerance = 1e-4  # 10e-5 in MATLAB is 1e-4
    
    # Equation (38): Compute the attitude profile matrix B.
    # v_b is 3 x n, v_i is 3 x n, and w is n-element vector.
    # Multiply each column of v_b by the corresponding weight.
    B = (v_b * w) @ v_i.T   # shape (3,3)
    
    # Equation (46): Compute vector Z.
    # MATLAB indices: Z = [B(2,3)-B(3,2); B(3,1)-B(1,3); B(1,2)-B(2,1)];
    Z = np.array([
        B[1, 2] - B[2, 1],
        B[2, 0] - B[0, 2],
        B[0, 1] - B[1, 0]
    ])
    
    # Equation (45): S = B + B'
    S = B + B.T
    
    # Equation (44): sigma = trace(B)
    sigma = np.trace(B)
    
    # Equation (63): 
    delta = np.linalg.det(S)
    # kappa = trace(delta*inv(S)) = delta * trace(inv(S))
    kappa = delta * np.trace(np.linalg.inv(S))
    
    # Equation (71):
    a = sigma**2 - kappa
    b = sigma**2 + (Z.T @ Z)
    c = delta + (Z.T @ S @ Z)
    d = Z.T @ (S @ S @ Z)
    constant = a * b + c * sigma - d
    
    # Characteristic equation (Eqn.70): 
    #   f(lambda) = lambda^4 - (a + b)*lambda^2 - c*lambda + constant = 0
    # Initial guess for lambda: sum(w)
    lam = np.sum(w)
    last_lam = 0.0
    
    # Newton-Raphson iteration
    while np.abs(lam - last_lam) >= tolerance:
        last_lam = lam
        f = lam**4 - (a + b)*lam**2 - c*lam + constant
        f_dot = 4*lam**3 - 2*(a + b)*lam - c
        lam = lam - f / f_dot
        
    # Eqn.66:
    omega = lam
    alpha = omega**2 - sigma**2 + kappa
    beta_val = omega - sigma  # beta is already used as quaternion variable in our code, so use beta_val.
    gamma = (omega + sigma) * alpha - delta
    
    # Equation (68): Calculate X = (alpha*I + beta*S + S^2)*Z
    I3 = np.eye(3)
    X = (alpha * I3 + beta_val * S + S @ S) @ Z
    
    # Equation (69): Construct optimal quaternion as [X; gamma] and normalize
    q_opt = np.concatenate((X, np.array([gamma])))
    norm_q = np.sqrt(gamma**2 + np.linalg.norm(X)**2)
    q_opt = q_opt / norm_q  # q_opt is [q0,q1,q2,q3], with q0 corresponding to gamma
    
    # Determine optimal rotation matrix from the optimal quaternion.
    C_opt = quaternion_to_DCM(q_opt)
    
    return C_opt, q_opt

In [14]:
# Concept Check 5 - QUEST
v_b = np.array([
    [0.8273, 0.5541, -0.0920],
    [-0.8285, 0.5522, -0.0955]
])

v_i = np.array([
    [-0.1517, -0.9669, 0.2050],
    [-0.8393, 0.4494, -0.3044]
])

# Call the QUEST function
DCM_estimated = quest(v_b, v_i, w)

# Print the resulting Direction Cosine Matrix (DCM)
print("Estimated DCM:")
print(np.round(DCM_estimated, 6))

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 

In [9]:
import time

# Define the test vectors
v_b = np.array([
    [0.8273, 0.5541, -0.0920],
    [-0.8285, 0.5522, -0.0955]
])

v_i = np.array([
    [-0.1517, -0.9669, 0.2050],
    [-0.8393, 0.4494, -0.3044]
])

# Measure time for Davenport's Q-Method
start_time_davenport = time.time()
DCM_davenport = davenport_q_method(v_b, v_i)
end_time_davenport = time.time()
time_davenport = end_time_davenport - start_time_davenport

# Print results for Davenport's Q-Method
print("Davenport's Q-Method:")
print("Estimated DCM:")
print(np.round(DCM_davenport, 6))
print(f"Time taken: {time_davenport:.12f} seconds\n")

# Measure time for QUEST algorithm
start_time_quest = time.time()
DCM_quest = quest_algorithm(v_b, v_i)
end_time_quest = time.time()
time_quest = end_time_quest - start_time_quest

# Print results for QUEST algorithm
print("QUEST Algorithm:")
print("Estimated DCM:")
print(np.round(DCM_quest, 6))
print(f"Time taken: {time_quest:.12f} seconds\n")


Davenport's Q-Method:
Estimated DCM:
[[ 0.415936 -0.854894  0.310087]
 [-0.833757 -0.494637 -0.245325]
 [ 0.363107 -0.156498 -0.918511]]
Time taken: 0.000000000000 seconds

QUEST Algorithm:
Estimated DCM:
[[ 0.415936 -0.854894  0.310087]
 [-0.833757 -0.494637 -0.245325]
 [ 0.363107 -0.156498 -0.918511]]
Time taken: 0.000000000000 seconds



# 4.6) OLAE

**<ins>目的</ins>**

- **Optimal Linear Attitude Estimator (OLAE)** は、宇宙機の姿勢決定において、Wahbaの問題を直接解くのとは異なる枠組みを提供する。
- 目標は、**古典的 Rodrigues パラメータ（CRPs）** を用い、**線形最小二乗法の定式化**により宇宙機の姿勢を推定することである：

  $$
  \mathbf{d} = [\mathbf{S}] \mathbf{q}
  $$

  ここで、
  - $\mathbf{q}$ は **CRPs** を表す。
  - $\mathbf{d}$ と $[\mathbf{S}]$ は、ボディ座標系と慣性座標系の観測間の関係を符号化する。

- OLAE は、固有値分解や反復的な根の求解法を回避し、**計算効率**を保持しながら姿勢決定を実現する。

---

**<ins>定式化</ins>**

1. **Cayley変換**  
   - OLAE法は、**Cayley変換**を用いて回転行列 $[\bar{\mathbf{B}}\mathbf{N}]$ をパラメータ化する：

     $$
     [\bar{\mathbf{B}}\mathbf{N}] = \left([\mathbf{I}_{3 \times 3}] + [\mathbf{\tilde{q}}]\right)^{-1} \left([\mathbf{I}_{3 \times 3}] - [\mathbf{\tilde{q}}]\right)
     $$

     ここで $[\mathbf{\tilde{q}}]$ は、Rodriguesパラメータ $\mathbf{q}$ の**スキュー対称行列**である。

<br>

2. **問題の線形化**  
   - 慣性座標系のベクトル $\mathcal{^N}\hat{v}_i$ に対し、ボディ座標系の対応するベクトルは

     $$
     \mathbf{^B}\hat{v}_i = [\bar{\mathbf{B}}\mathbf{N}] \mathcal{^N}\hat{v}_i
     $$

     で表される。
   - ここで、Cayley変換を代入すると、

     $$
     \mathbf{^B}\hat{v}_i = \left([\mathbf{I}_{3 \times 3}] + [\mathbf{\tilde{q}}]\right)^{-1} \left([\mathbf{I}_{3 \times 3}] - [\mathbf{\tilde{q}}]\right) \mathcal{^N}\hat{v}_i
     $$

   - 両辺に $\left([\mathbf{I}_{3 \times 3}] + [\mathbf{\tilde{q}}]\right)$ を掛けると、

     $$
     \left([\mathbf{I}_{3 \times 3}] + [\mathbf{\tilde{q}}]\right) \mathbf{^B}\hat{v}_i = \left([\mathbf{I}_{3 \times 3}] - [\mathbf{\tilde{q}}]\right) \mathcal{^N}\hat{v}_i
     $$

     項展開すると、

     $$
     \mathbf{^B}\hat{v}_i + [\mathbf{\tilde{q}}]\mathbf{^B}\hat{v}_i = \mathcal{^N}\hat{v}_i - [\mathbf{\tilde{q}}]\mathcal{^N}\hat{v}_i
     $$

   - 整理して $\mathbf{^B}\hat{v}_i - \mathcal{^N}\hat{v}_i$ を孤立させると、

     $$
     \mathbf{^B}\hat{v}_i - \mathcal{^N}\hat{v}_i = -[\mathbf{\tilde{q}}] \left(\mathbf{^B}\hat{v}_i + \mathcal{^N}\hat{v}_i\right)
     $$

     となり、この形はさらに単純化のために線形化される。

   - 以下を定義する：

     $$
     \mathbf{s}_i = \mathbf{^B}\hat{v}_i + \mathcal{^N}\hat{v}_i, \quad \mathbf{d}_i = \mathbf{^B}\hat{v}_i - \mathcal{^N}\hat{v}_i
     $$

     $\mathbf{s}_i$ と $\mathbf{d}_i$ は、ベクトルの和と差を記述する便宜的な変数であって、特定の幾何学的意味を持つものではない。

   - 定義を代入すると、

     $$
     \mathbf{d}_i = -[\mathbf{\tilde{q}}] \mathbf{s}_i
     $$

   - さらに、望ましい線形形に合わせるため、

     $$
     \mathbf{d}_i = [\mathbf{\tilde{s}}_i] \mathbf{q}
     $$

     と書く。ここで $[\mathbf{\tilde{s}}_i]$ は、$\mathbf{s}_i$ のスキュー対称行列である。

<br>

3. **行列形式**  
   - $N$ 個の観測について、$\mathbf{d}_i$ と $[\mathbf{\tilde{s}}_i]$ を次のように行列として積み重ねる：

     $$
     \mathbf{d} = \begin{bmatrix} \mathbf{d}_1 \\ \mathbf{d}_2 \\ \vdots \\ \mathbf{d}_N \end{bmatrix}, \quad
     [\mathbf{S}] = \begin{bmatrix} [\mathbf{\tilde{s}}_1] \\ [\mathbf{\tilde{s}}_2] \\ \vdots \\ [\mathbf{\tilde{s}}_N] \end{bmatrix}
     $$

   - これにより、線形システムは

     $$
     \mathbf{d} = [\mathbf{S}] \mathbf{q}
     $$

     となる。

<br>

4. **重み付き最小二乗解**  
   - 観測の重み $w_i$ を取り入れるため、対角の重み行列を構築する：

     $$
     [\mathbf{W}] = 
     \begin{bmatrix}
     w_1 \mathbf{I}_{3 \times 3} & \mathbf{0}_{3 \times 3} & \cdots & \mathbf{0}_{3 \times 3} \\
     \mathbf{0}_{3 \times 3} & w_2 \mathbf{I}_{3 \times 3} & \cdots & \mathbf{0}_{3 \times 3} \\
     \vdots & \vdots & \ddots & \vdots \\
     \mathbf{0}_{3 \times 3} & \mathbf{0}_{3 \times 3} & \cdots & w_N \mathbf{I}_{3 \times 3}
     \end{bmatrix}
     $$

     これは、$\text{diag}$ 演算子を用いて次のように書ける：

     $$
     [\mathbf{W}] = \text{diag}\left(w_1 \mathbf{I}_{3 \times 3}, w_2 \mathbf{I}_{3 \times 3}, \dots, w_N \mathbf{I}_{3 \times 3}\right)
     $$

   - 重み付き最小二乗解を用いて CRPs $\mathbf{q}$ を求める：

     $$
     \mathbf{q} = \left([\mathbf{S}]^T [\mathbf{W}] [\mathbf{S}]\right)^{-1} [\mathbf{S}]^T [\mathbf{W}] \mathbf{d}
     $$

---

**<ins>OLAE の利点</ins>**

1. **計算効率**  
   - 固有値分解や反復的な根の求解法を回避する。
   - 完全に線形化された定式化により、行列積と逆行列計算のみで済む。

2. **重みの柔軟性**  
   - 各観測の精度や信頼性を考慮した**観測重み**を明示的に取り入れる。

3. **単純さ**  
   - 線形最小二乗法の定式化により、実装が容易である。

4. **リアルタイム応用**  
   - 計算資源が限られる状況下やリアルタイムな姿勢決定に適している。

---

**<ins>Wahba の問題との比較</ins>**

| **項目**                  | **Wahba の問題 (QUEST)**                   | **OLAE**                                       |
|---------------------------|--------------------------------------------|------------------------------------------------|
| **目的**                  | 二次のコスト関数を最小化する。             | 線形最小二乗問題として定式化する。             |
| **パラメータ化**          | クォータニオンまたは古典的 Rodrigues パラメータ。 | 古典的 Rodrigues パラメータ (CRPs)。           |
| **解法**                  | 固有値分解または反復的最適化。               | 重み付き最小二乗解。                           |
| **計算複雑性**            | 固有値計算により高い。                      | 線形定式化により低い。                          |

---

**<ins>まとめ</ins>**

- **Optimal Linear Attitude Estimator (OLAE)** は、Wahba の問題を解く従来の手法に代わる選択肢である。
- **Cayley変換** を用い、問題を **古典的 Rodrigues パラメータ (CRPs)** を基に線形的に定式化する。
- 解は **重み付き最小二乗法** により求められ、ボディ座標系の観測と慣性座標系の既知ベクトル間の誤差を最小化する。
- OLAE は計算効率に優れ、リアルタイムの応用や計算資源の限られたシステムに適している。
- QUEST や Davenport の Q-Method とは異なり、非線形最適化や固有値分解を回避するため、姿勢決定においてより単純かつ高速な代替手法を提供する。

In [10]:
def olae_method(B_v_k, N_v_k, weights=None):
    """
    Implements the Optimal Linear Attitude Estimator (OLAE) to estimate the rotation matrix (DCM)
    from inertial frame to body frame using Classical Rodrigues Parameters (CRPs).

    Args:
        B_v_k (numpy.ndarray): Array of shape (N, 3) containing N body-frame vectors \( \mathbf{^B}\hat{v}_k \).
                               Each row corresponds to the k-th body-frame vector.

        N_v_k (numpy.ndarray): Array of shape (N, 3) containing N inertial-frame vectors \( \mathcal{^N}\hat{v}_k \).
                               Each row corresponds to the k-th inertial-frame vector.

        weights (numpy.ndarray, optional): Array of shape (N,) containing weights \( w_k \) for each vector pair.
                                           If None, equal weights are assumed.

    Returns:
        numpy.ndarray: Estimated rotation matrix (3x3).
    """
    # Ensure inputs are NumPy arrays
    B_v_k = np.asarray(B_v_k, dtype=float)
    N_v_k = np.asarray(N_v_k, dtype=float)

    # Validate input dimensions
    if B_v_k.shape != N_v_k.shape:
        raise ValueError("Input vector arrays must have the same shape.")

    # Number of vector observations
    N = B_v_k.shape[0]

    # Validate each vector in B_v_k and N_v_k
    for k in range(N):
        validate_vec3(B_v_k[k])
        validate_vec3(N_v_k[k])

    # Assign equal weights if none provided
    if weights is None:
        weights = np.ones(N)
    else:
        weights = np.asarray(weights, dtype=float)
        if weights.shape[0] != N:
            raise ValueError("Weights array must have the same length as the number of vector observations.")

    # Normalize the input vectors to unit length
    B_v_k /= np.linalg.norm(B_v_k, axis=1, keepdims=True)
    N_v_k /= np.linalg.norm(N_v_k, axis=1, keepdims=True)

    # Initialize lists to collect d_i and S_i
    d_list = []
    S_list = []

    # Compute d_i and S_i for each observation
    for k in range(N):
        d_i = B_v_k[k] - N_v_k[k]  # Difference vector
        s_i = B_v_k[k] + N_v_k[k]  # Sum vector
        s_tilde_i = skew_symmetric(s_i)  # Skew-symmetric matrix of s_i
        d_list.append(d_i)
        S_list.append(s_tilde_i)

    # Stack d_i and S_i to form d and S matrices
    d = np.concatenate(d_list)         # Shape: (3N,)
    S = np.vstack(S_list)              # Shape: (3N, 3)

    # Construct the weight matrix W as a block diagonal matrix using numpy
    W = np.zeros((3*N, 3*N))
    for i in range(N):
        idx = slice(3*i, 3*(i+1))
        W[idx, idx] = weights[i] * np.eye(3)

    # Solve for CRP q using the weighted least squares solution
    q = np.linalg.multi_dot((np.linalg.inv(np.linalg.multi_dot((S.T, W, S))), S.T, W, d)).reshape((3,))

    # Convert CRP q to DCM
    C = CRP_to_DCM(q)

    return C


In [11]:
# Concept Check 5 - QUEST
v_b = np.array([
    [0.8273, 0.5541, -0.0920],
    [-0.8285, 0.5522, -0.0955]
])

v_i = np.array([
    [-0.1517, -0.9669, 0.2050],
    [-0.8393, 0.4494, -0.3044]
])

# Call the QUEST function
DCM_estimated = olae_method(v_b, v_i)

# Print the resulting Direction Cosine Matrix (DCM)
print("Estimated DCM:")
print(np.round(DCM_estimated, 6))

Estimated DCM:
[[ 0.416219 -0.854762  0.31007 ]
 [-0.833608 -0.494901 -0.245298]
 [ 0.363125 -0.156379 -0.918524]]
